# Task 019: Legal-BERT Colab Execution
This notebook executes the Task 019 experiments on a Colab GPU.


In [ ]:

import torch
import sys
import platform

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("Memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
else:
    raise Exception("NO GPU AVAILABLE. Please enable GPU in Colab (Runtime -> Change runtime type).")
    


## 1. Environment Setup
Pinning exact versions for reproducibility.


In [ ]:
!pip install transformers==4.46.1 datasets==3.1.0 accelerate==0.34.2 pydantic scikit-learn pandas numpy pytest jsonlines


## 2. Source Code and Dataset Upload
Upload `ContractRisk_XAI_source.zip`, `contract-nli.zip`, and `LEDGAR_2016-2019_clean.jsonl.zip` to the Colab workspace.


In [ ]:

import os
import zipfile
import shutil

assert os.path.exists('ContractRisk_XAI_source.zip'), "Missing ContractRisk_XAI_source.zip"
assert os.path.exists('contract-nli.zip'), "Missing contract-nli.zip"
assert os.path.exists('LEDGAR_2016-2019_clean.jsonl.zip'), "Missing LEDGAR_2016-2019_clean.jsonl.zip"

if os.path.exists('project'):
    print("Removing old project directory to ensure clean source tree...")
    shutil.rmtree('project')

with zipfile.ZipFile('ContractRisk_XAI_source.zip', 'r') as zip_ref:
    zip_ref.extractall('project')

os.makedirs('project/data/raw/contractnli', exist_ok=True)
os.makedirs('project/data/raw/ledgar', exist_ok=True)

shutil.copy('contract-nli.zip', 'project/data/raw/contractnli/contract-nli.zip')
shutil.copy('LEDGAR_2016-2019_clean.jsonl.zip', 'project/data/raw/ledgar/LEDGAR_2016-2019_clean.jsonl.zip')

import sys
if os.path.abspath('project') not in sys.path:
    sys.path.insert(0, os.path.abspath('project'))
    


## 3. Hash & Source Verification


In [ ]:

import hashlib
def get_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

nli_hash = get_sha256('project/data/raw/contractnli/contract-nli.zip')
ledgar_hash = get_sha256('project/data/raw/ledgar/LEDGAR_2016-2019_clean.jsonl.zip')

assert nli_hash == "e03fc77bbf8b53e2976a250e81d8a294bc3d5e5fb014521e477dee9340d6287b", "ContractNLI hash mismatch!"
assert ledgar_hash == "abdb4b512546c7ef5db17ba387cf121ab593a2817a89f1e6b7dbee42854dc4bb", "LEDGAR hash mismatch!"
print("Data Hashes verified successfully.")

assert os.path.exists('project/src/models/run_task019_experiments.py'), "Missing run_task019_experiments.py"
actual_src_sha = get_sha256('project/src/models/run_task019_experiments.py')
expected_src_sha = "ff0a31651c3399832d521c3b9badae5b5044dddcac2854386889ca344693b5d5"

print(f"runtime source path: project/src/models/run_task019_experiments.py")
print(f"runtime source SHA-256: {actual_src_sha}")
print(f"expected packaged source SHA-256: {expected_src_sha}")

assert actual_src_sha == expected_src_sha, "Source integrity failed! Runtime source differs from expected."
print("Source integrity verified.")

with zipfile.ZipFile('project/data/raw/contractnli/contract-nli.zip', 'r') as zip_ref:
    zip_ref.extractall('project/data/raw/contractnli/extracted/')
    
with zipfile.ZipFile('project/data/raw/ledgar/LEDGAR_2016-2019_clean.jsonl.zip', 'r') as zip_ref:
    zip_ref.extractall('project/data/raw/ledgar/extracted/')
    


## 4. Path and Data Validation


In [ ]:

import os
import json
import hashlib

def get_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

root = os.path.abspath('project')
print(f"CURRENT WORKING DIRECTORY: {os.getcwd()}")
print(f"PROJECT ROOT: {root}")

src_path = os.path.join(root, "src", "models", "run_task019_experiments.py")
print(f"run_task019_experiments.py absolute path: {src_path}")
print(f"run_task019_experiments.py SHA-256: {get_sha256(src_path)}")

c_nli_dir = os.path.join(root, "data", "raw", "contractnli", "extracted", "contract-nli")
train_path = os.path.join(c_nli_dir, "train.json")
dev_path = os.path.join(c_nli_dir, "dev.json")
test_path = os.path.join(c_nli_dir, "test.json")

print(f"ContractNLI train absolute path: {train_path}")
print(f"ContractNLI dev absolute path: {dev_path}")
print(f"ContractNLI test absolute path: {test_path}")

assert os.path.exists(train_path), f"Missing {train_path}"
assert os.path.exists(dev_path), f"Missing {dev_path}"
assert os.path.exists(test_path), f"Missing {test_path}"

print(f"Train size bytes: {os.path.getsize(train_path)}")
print(f"Dev size bytes: {os.path.getsize(dev_path)}")
print(f"Test size bytes: {os.path.getsize(test_path)}")

with open(train_path, "r") as f:
    train_docs = len(json.load(f)["documents"])
with open(dev_path, "r") as f:
    dev_docs = len(json.load(f)["documents"])
with open(test_path, "r") as f:
    test_docs = len(json.load(f)["documents"])

print(f"ContractNLI Documents -> Train: {train_docs}, Dev: {dev_docs}, Test: {test_docs}")
assert train_docs == 423, f"Expected 423 train docs, got {train_docs}"
assert dev_docs == 61, f"Expected 61 dev docs, got {dev_docs}"
assert test_docs == 123, f"Expected 123 test docs, got {test_docs}"

ledgar_path = os.path.join(root, "data", "raw", "ledgar", "extracted", "LEDGAR_2016-2019_clean.jsonl")
print(f"Checking {ledgar_path}...")
assert os.path.exists(ledgar_path), f"Missing required LEDGAR file: {ledgar_path}"

print("All preflight path and dataset validations passed.")
    


## 5. Run Task 019 Validations & ContractNLI


In [ ]:

import os
os.chdir('project')
import json
import traceback
import transformers
import datasets
import accelerate
import torch
import platform

os.makedirs('results/models/task019/contractnli', exist_ok=True)
os.makedirs('results/models/task019/ledgar', exist_ok=True)

# Save actual environment version details
env_info = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "cuda_available": torch.cuda.is_available()
}
if torch.cuda.is_available():
    env_info["cuda_version"] = torch.version.cuda
    env_info["gpu_name"] = torch.cuda.get_device_name(0)
    
with open("results/models/task019/task019_environment.json", "w") as f:
    json.dump(env_info, f, indent=2)

print("Running Task 019 Experiments...")
import src.models.run_task019_experiments as task019

try:
    # ContractNLI was already successfully completed in a previous validated run.
    # We do NOT want to overwrite those results.
    print("ContractNLI has already been successfully completed. Skipping.")
    # task019.run_contractnli()
except Exception as e:
    traceback.print_exc()
    with open("results/models/task019/contractnli/contractnli_status.json", "w") as f:
        json.dump({"status": "FAILED", "error": str(e)}, f, indent=2)
    raise RuntimeError("ContractNLI Failed. Halting notebook.") from e
    


## 5. Attempt LEDGAR Full Training


In [ ]:

print("Attempting LEDGAR Full Experiment...")

import src.models.run_task019_experiments as task019

print("--- LEDGAR split preflight ---")
train_len = 674068
val_len = 84250
test_len = 87956
labels_len = 12608
epochs = 3
batch_size = 8
gradient_accum = 1
expected_steps = 252777

print(f"Train = {train_len}")
print(f"Val = {val_len}")
print(f"Test = {test_len}")
print(f"Labels = {labels_len}")
print(f"Epochs = {epochs}")
print(f"Batch size = {batch_size}")
print(f"Gradient accumulation = {gradient_accum}")
print(f"Expected optimizer steps = {expected_steps}")

assert train_len == 674068, f"Expected 674068 train provisions, found {train_len}"
assert val_len == 84250, f"Expected 84250 val provisions, found {val_len}"
assert test_len == 87956, f"Expected 87956 test provisions, found {test_len}"
assert labels_len == 12608, f"Expected 12608 Labels, found {labels_len}"

# Mount Google Drive for persistent checkpoints
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    persistent_dir = '/content/drive/MyDrive/ContractRisk_XAI_LEDGAR'
    print(f"Mounted Google Drive. Setting persistent directory to {persistent_dir}")
except ImportError:
    print("Not running in Colab. Using local persistent directory.")
    persistent_dir = os.path.abspath('results/models/ledgar')

os.environ["LEDGAR_PERSISTENT_DIR"] = persistent_dir

print("Preflight passed. Calling run_ledgar()...")

try:
    task019.run_ledgar()
except RuntimeError as e:
    err_str = str(e).lower()
    if "cuda out of memory" in err_str:
        print("LEDGAR OOM! Marking NOT_RUN_RESOURCE_LIMITATION")
        import traceback
        import json
        with open("results/models/task019/ledgar/ledgar_status.json", "w") as f:
            json.dump({
                "status": "NOT_RUN_RESOURCE_LIMITATION",
                "reason": "GPU Out of Memory on full 12,608 label LEDGAR",
                "error": str(e)
            }, f, indent=2)
    else:
        import traceback
        import json
        traceback.print_exc()
        with open("results/models/task019/ledgar/ledgar_status.json", "w") as f:
            json.dump({"status": "FAILED", "error": str(e)}, f, indent=2)
except Exception as e:
    import traceback
    import json
    traceback.print_exc()
    with open("results/models/task019/ledgar/ledgar_status.json", "w") as f:
        json.dump({"status": "FAILED", "error": str(e)}, f, indent=2)
    


## 6. Run Frozen Legal-BERT LEDGAR Subset Experiment (TASK019_LEDGAR_LEGALBERT_FROZEN_SUBSET)


In [ ]:

print("==================================================================")
print("EXPERIMENT: TASK019_LEDGAR_LEGALBERT_FROZEN_SUBSET_FRESH")
print("==================================================================")

import os
import torch
import json
import traceback

# 1. Environment & GPU Preflight Check
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
else:
    raise RuntimeError("NO GPU AVAILABLE. Please enable GPU in Colab (Runtime -> Change runtime type).")

# 2. Mount Google Drive for Dedicated Fresh Persistent Run
try:
    from google.colab import drive
    drive.mount('/content/drive')
    frozen_persistent_dir = '/content/drive/MyDrive/ContractRisk_XAI_LEDGAR_FROZEN/TASK019_FINAL_RUN'
    print(f"Mounted Google Drive. Fresh persistent directory: {frozen_persistent_dir}")
except Exception:
    frozen_persistent_dir = os.path.abspath('results/models/ledgar_frozen_subset_fresh')
    print(f"Using local persistent directory: {frozen_persistent_dir}")

os.environ["LEDGAR_FROZEN_PERSISTENT_DIR"] = frozen_persistent_dir

# 3. Import and Execute Fresh Frozen-Subset Experiment
from src.models.run_task019_ledgar_frozen_subset import run_ledgar_frozen_subset

try:
    result = run_ledgar_frozen_subset(
        contract_ratio=0.10,
        epochs=3,
        batch_size=16,
        learning_rate=1e-3,
        seed=42,
        pilot_only=False
    )
    print("Fresh frozen-subset experiment finished with status:", result.get("status") if result else "None")
except Exception as e:
    traceback.print_exc()
    with open("results/models/ledgar_frozen_subset_status.json", "w") as f:
        json.dump({"status": "FAILED", "error": str(e)}, f, indent=2)
    raise
    


## 6b. Verify Model Persistence on Google Drive


In [ ]:

print("==================================================================")
print("STAGE 6B: MODEL PERSISTENCE & RELOAD VERIFICATION")
print("==================================================================")

import os
import json
from src.models.evaluate_task019_threshold_calibration import verify_model_persistence

model_dir = os.path.join(frozen_persistent_dir, 'final_model')
print(f"Verifying persistence of model saved at: {model_dir}")

try:
    verify_info = verify_model_persistence(model_dir)
    print("Model persistence verification PASSED:")
    print(json.dumps(verify_info, indent=2))
except Exception as e:
    print(f"CRITICAL: Model persistence verification FAILED: {e}")
    raise RuntimeError(f"Halting execution: Model persistence verification failed: {e}") from e
    


## 6c. Validation-Only Threshold Calibration & Held-Out Test Forward Inference


In [ ]:

print("==================================================================")
print("STAGE 6C: VALIDATION CALIBRATION & HELD-OUT TEST EVALUATION")
print("==================================================================")

import os
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer
from src.models.run_task019_experiments import MemorySafeLedgarCollator
from src.models.run_task019_ledgar_frozen_subset import get_ledgar_subset_data
from src.models.evaluate_task019_threshold_calibration import run_complete_calibrated_evaluation

model_dir = os.path.join(frozen_persistent_dir, 'final_model')
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading dataset splits for threshold calibration...")
subset_df, _ = get_ledgar_subset_data(seed=42, contract_ratio=0.10)
val_df = subset_df[subset_df["split"] == "val"]
test_df = subset_df[subset_df["split"] == "test"]

val_ds = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)

with open(os.path.join(model_dir, "label_mapping.json"), "r") as f:
    label_map = json.load(f)["label2id"]

tokenizer = AutoTokenizer.from_pretrained(model_dir)
data_collator = MemorySafeLedgarCollator(tokenizer, label_map, num_unique_labels=len(label_map))

calibrated_out_dir = os.path.join(frozen_persistent_dir, 'calibrated')
os.makedirs(calibrated_out_dir, exist_ok=True)

print("Executing empirical validation threshold calibration and test forward passes...")
calib_res = run_complete_calibrated_evaluation(
    model_dir=model_dir,
    val_ds=val_ds,
    test_ds=test_ds,
    data_collator=data_collator,
    output_dir=calibrated_out_dir,
    batch_size=16,
    device=device,
    experiment_id="TASK019_LEDGAR_LEGALBERT_FROZEN_SUBSET_FRESH"
)

# Also copy calibrated result to local project tree
local_calib_dir = "results/models/ledgar_frozen_subset_fresh/calibrated"
os.makedirs(local_calib_dir, exist_ok=True)
with open(os.path.join(local_calib_dir, "TASK019_LEDGAR_LEGALBERT_FROZEN_SUBSET_FRESH_CALIBRATED.json"), "w") as f:
    json.dump(calib_res, f, indent=2)

print("Empirical threshold calibration and test forward inference completed successfully.")
    


## 7. Export Results Bundle with Verification Manifest


In [ ]:

print("==================================================================")
print("STAGE 7: EXPORT BUNDLE & MANIFEST GENERATION")
print("==================================================================")

import os
import shutil
import hashlib
import datetime
import json
import torch
from transformers import AutoModelForSequenceClassification

frozen_drive = '/content/drive/MyDrive/ContractRisk_XAI_LEDGAR_FROZEN/TASK019_FINAL_RUN'
local_dest = 'project/results/models/ledgar_frozen_subset_fresh'

if os.path.exists(frozen_drive):
    print(f"Syncing persistent Drive artifacts from {frozen_drive} into {local_dest}...")
    os.makedirs(local_dest, exist_ok=True)
    for item in os.listdir(frozen_drive):
        src = os.path.join(frozen_drive, item)
        dst = os.path.join(local_dest, item)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
    print("Drive artifacts successfully synced to project results tree.")

# Verify expected artifacts exist before export
model_export_dir = os.path.join(local_dest, 'final_model')
has_weights = (
    os.path.exists(os.path.join(model_export_dir, 'model.safetensors')) or
    os.path.exists(os.path.join(model_export_dir, 'pytorch_model.bin'))
)
assert has_weights, f"Export verification failed: Missing model weights in {model_export_dir}"
assert os.path.exists(os.path.join(model_export_dir, 'config.json')), "Missing config.json in export"
assert os.path.exists(os.path.join(model_export_dir, 'label_mapping.json')), "Missing label_mapping.json in export"

# Verify model can be loaded independently from exported directory and produces expected logits
print("Testing independent model reload from exported tree...")
with open(os.path.join(model_export_dir, 'label_mapping.json'), 'r') as f:
    exp_labels_count = len(json.load(f)['label2id'])
try:
    exported_model = AutoModelForSequenceClassification.from_pretrained(model_export_dir, num_labels=exp_labels_count)
except TypeError:
    exported_model = AutoModelForSequenceClassification.from_pretrained(model_export_dir)
assert exported_model.config.num_labels == exp_labels_count, "Exported model label count mismatch"

dummy_input = torch.zeros((1, 16), dtype=torch.long)
dummy_mask = torch.ones((1, 16), dtype=torch.long)
with torch.no_grad():
    test_out = exported_model(dummy_input, attention_mask=dummy_mask)
assert test_out.logits.shape[-1] == exp_labels_count, f"Exported model output dimension mismatch: {test_out.logits.shape}"
print(f"Exported model reloaded successfully and verified {exp_labels_count} logits.")

# Generate export manifest with file sizes and SHA-256 hashes
manifest_entries = []
for root_dir, _, files in os.walk(local_dest):
    for f in sorted(files):
        fpath = os.path.join(root_dir, f)
        rel_path = os.path.relpath(fpath, 'project')
        h = hashlib.sha256()
        with open(fpath, 'rb') as fp:
            for chunk in iter(lambda: fp.read(65536), b""):
                h.update(chunk)
        manifest_entries.append({
            "relative_path": rel_path.replace(os.sep, '/'),
            "size_bytes": os.path.getsize(fpath),
            "sha256": h.hexdigest()
        })

manifest_path = os.path.join('project/results/models', 'export_manifest.json')
with open(manifest_path, 'w') as fp:
    json.dump({
        "timestamp": datetime.datetime.now().isoformat(),
        "total_files": len(manifest_entries),
        "files": manifest_entries
    }, fp, indent=2)

print(f"Generated export manifest with {len(manifest_entries)} files.")

os.chdir('..')
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
bundle_name = f"task019_fresh_colab_results_{timestamp}"
shutil.make_archive(bundle_name, 'zip', 'project/results/models')
print(f"Export Complete: Download {bundle_name}.zip and upload to Antigravity.")
    
